In [ ]:
import polars as pl

iot_malware_df = pl.read_parquet("processed/iot_malware_dataset.parquet")
unsw_iot_df = pl.read_parquet("processed/unsw_iot_dataset.parquet")
cic_iot_df = pl.read_parquet("processed/cic_iot_dataset2022.parquet")

common_columns = sorted(
    set(iot_malware_df.columns)
    & set(unsw_iot_df.columns)
    & set(cic_iot_df.columns)
)
print(common_columns)

print(iot_malware_df["source_ip"])
print(unsw_iot_df["source_ip"])
print(cic_iot_df["source_ip"])

['avg_packet_size', 'dataset_source', 'device_id', 'device_type', 'dst_port', 'flow_duration_ms', 'manufacturer', 'packet_rate', 'src_port', 'timestamp', 'total_bytes', 'total_packets', 'transport_protocol']


: 

In [ ]:
import polars as pl

for name, path in {
    "IoT Malware": "processed/iot_malware_dataset.parquet",
    "UNSW IoT": "processed/unsw_iot_dataset.parquet",
    "CIC IoT 2022": "processed/cic_iot_dataset2022.parquet",
}.items():
    counts = (
        pl.scan_parquet(path)
        .group_by("attack_type")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    print(f"{name}:")
    with pl.Config(tbl_rows=-1):
        print(counts)
    nonnull_count = counts.filter(pl.col("attack_type").is_not_null())["count"].sum()
    print(f"Total non-null attack_type: {nonnull_count}")
    print()